# Selection bias $b_\text{sel}(\theta)$ vs. $\lambda^\text{ob} - \lambda^\text{tr}$

At fixed $(\lambda^\text{ob}, z^\text{ob})$, the projection-effects
selection bias interpolates between two asymptotes via a sigmoid in
angular separation $\theta$:

$$b_\text{sel}(\theta \mid \lambda^\text{ob}, \lambda^\text{tr}, z^\text{ob}) = b^\text{small} (1 - \sigma(\theta)) + b^\text{large} \sigma(\theta).$$

Both $b^\text{small}$ and $b^\text{large}$ depend on $\lambda^\text{ob} - \lambda^\text{tr}$ through the projection
bias $\Delta^\text{prj}$. Here we sweep $\lambda^\text{tr}$ for a fixed $\lambda^\text{ob}=20$,
$z^\text{ob}=0.5$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from richness_selection import (
    Cosmology, PkGrid, HMF, Bias, MOR, NFWMiscentered, SelBias,
)
from richness_selection.sigma_m import SigmaM

cosmo    = Cosmology()
pk       = PkGrid(cosmo)
sigma_m  = SigmaM(pk)
hmf      = HMF(sigma_m)
bias     = Bias(sigma_m)
mor      = MOR()
sel_bias = SelBias(cosmo, pk, hmf, bias, mor)

In [ ]:
lob  = 20.0
zob  = 0.5

pre  = sel_bias.bias_precompute(lob, zob)
print('bias_precompute (ltr-invariant):')
for k, v in pre.items():
    print(f'  {k:>6s} = {v:+.4e}')

In [ ]:
# Sweep lambda^ob - lambda^tr at fixed lob.
deltas = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 5.0])     # lob - ltr
ltrs   = lob - deltas

theta_lam = sel_bias._theta_tgt(lob, zob)
thetas    = np.linspace(1e-5, 2.0 * theta_lam, 200)

fig, ax = plt.subplots(figsize=(7, 4.5))
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(ltrs)))

for col, d, ltr in zip(cmap, deltas, ltrs):
    b = np.array([sel_bias.b_sel_of_theta(th, lob, zob, ltr, precomp=pre)
                  for th in thetas])
    ax.plot(thetas / theta_lam, b, color=col, lw=1.8,
            label=f'$\\lambda^{{ob}}-\\lambda^{{tr}} = {d:+.1f}$')

ax.axhline(pre['bhalo'], color='0.5', lw=1.0, ls='--',
           label=f'$b_\\mathrm{{halo}} = {pre["bhalo"]:.2f}$')
ax.set_xlabel(r'$\theta / \theta_{\lambda^\mathrm{ob}}$')
ax.set_ylabel(r'$b_\mathrm{sel}(\theta \,|\, \lambda^\mathrm{ob},\, \lambda^\mathrm{tr},\, z^\mathrm{ob})$')
ax.set_title(rf'$\lambda^\mathrm{{ob}}={lob:.0f}$, $z^\mathrm{{ob}}={zob}$')
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

## Marginalised over $\lambda^\text{tr}$

The observable in the likelihood is $\langle b_\text{sel}(\theta) \rangle_{p(\lambda^\text{tr} | \lambda^\text{ob})}$.

In [ ]:
bsel_marg = sel_bias.b_sel_marginalised(thetas, lob, zob, precomp=pre)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(thetas / theta_lam, bsel_marg, color='C0', lw=2)
ax.axhline(pre['bhalo'], color='0.5', lw=1.0, ls='--',
           label=f'$b_\\mathrm{{halo}} = {pre["bhalo"]:.2f}$')
ax.set_xlabel(r'$\theta / \theta_{\lambda^\mathrm{ob}}$')
ax.set_ylabel(r'$\langle b_\mathrm{sel}(\theta)\rangle_{\lambda^\mathrm{tr}}$')
ax.set_title(r'$\lambda^\mathrm{tr}$-marginalised selection bias')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()